# Verify Fitting Approach — SLP MNIST

Three independent checks that the per-run fitting and averaging in `fitting_function_IPA.ipynb` is correct.
Reads only already-saved CSVs — no re-fitting needed, original notebook untouched.

| Cell | What it checks |
|------|----------------|
| **A** | Run count audit — how many unique runs are in each saved CSV vs expected 100 |
| **B** | Visual spot-check — overlay all individual fitted curves + mean for one (P%, BS) |
| **C** | Numerical cross-check — re-derive mean from CSV and compare to saved `mean_fit` |

In [1]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Paths (must match fitting_function_IPA.ipynb) ────────────────────────────
OUT_DIR = r"C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\Fitting_IPA_curves_data"

BATCH_SIZES = [64, 1024, 60000]

# ── Auto-detect pruning levels from saved CSVs ───────────────────────────────
sample_csvs = glob.glob(os.path.join(OUT_DIR, "BS_64", "per_run_fits_p_*_bs_64.csv"))
PRUNING_LEVELS = sorted([
    float(re.search(r"per_run_fits_p_(.*?)_bs_64", os.path.basename(f)).group(1))
    for f in sample_csvs
])
print(f"Found {len(PRUNING_LEVELS)} pruning levels: {PRUNING_LEVELS}")
print(f"OUT_DIR exists: {os.path.isdir(OUT_DIR)}")

Found 0 pruning levels: []
OUT_DIR exists: False


## Cell A — Run Count Audit

Reads each `per_run_fits_p_{p}_bs_{bs}.csv` and counts unique `Run` IDs.
Prints a full table and highlights any (P%, BS) with fewer than 100 runs.

- **100 runs** → all raw data present, fitting covered everything
- **< 100 runs** → raw `.txt` files were never generated for those configs (data gap, not a code bug)

In [2]:
# ── Cell A — Run Count Audit ─────────────────────────────────────────────────

rows = []
missing_csvs = []

for bs in BATCH_SIZES:
    for p in PRUNING_LEVELS:
        csv_path = os.path.join(OUT_DIR, f"BS_{bs}", f"per_run_fits_p_{p}_bs_{bs}.csv")

        if not os.path.isfile(csv_path):
            missing_csvs.append((p * 100, bs))
            continue

        df = pd.read_csv(csv_path)
        n_runs   = df["Run"].nunique()
        n_rows   = len(df)
        rows.append({
            "P%":        p * 100,
            "BS":        bs,
            "runs":      n_runs,
            "total_rows": n_rows,
            "ok":        "YES" if n_runs == 100 else f"ONLY {n_runs}"
        })

audit_df = pd.DataFrame(rows)

print("=" * 55)
print("  Full audit table")
print("=" * 55)
print(audit_df.to_string(index=False))

incomplete = audit_df[audit_df["runs"] < 100]
print(f"\n{'='*55}")
if incomplete.empty:
    print("  All (P%, BS) combinations have 100 runs — no gaps.")
else:
    print(f"  {len(incomplete)} combinations with < 100 runs (raw data not generated):")
    print(incomplete[["P%", "BS", "runs"]].to_string(index=False))

if missing_csvs:
    print(f"\n  {len(missing_csvs)} CSV files not found at all:")
    for p_pct, bs in missing_csvs:
        print(f"    P%={p_pct}  BS={bs}")

  Full audit table
Empty DataFrame
Columns: []
Index: []


KeyError: 'runs'

## Cell B — Visual Spot-Check: All Individual Fits + Mean

For a chosen `(P_CHECK, BS_CHECK)`, plots:
- **Faint blue lines** — each of the 100 individual fitted curves
- **Bold red line** — the saved mean fit (`mean_fit_p_{p}_bs_{bs}.csv`)
- **Red shading** — ±1 std band

The mean should sit visually in the centre of the cloud.
Change `P_CHECK` and `BS_CHECK` to inspect any combination.

In [ ]:
# ── Cell B — Visual spot-check ───────────────────────────────────────────────

P_CHECK  = 0.0   # ← change to any pruning level, e.g. 0.8, 0.9
BS_CHECK = 64    # ← change to 64, 1024, or 60000

per_run_csv = os.path.join(OUT_DIR, f"BS_{BS_CHECK}", f"per_run_fits_p_{P_CHECK}_bs_{BS_CHECK}.csv")
mean_csv    = os.path.join(OUT_DIR, f"BS_{BS_CHECK}", f"mean_fit_p_{P_CHECK}_bs_{BS_CHECK}.csv")

assert os.path.isfile(per_run_csv), f"Missing: {per_run_csv}"
assert os.path.isfile(mean_csv),    f"Missing: {mean_csv}"

per_run_df = pd.read_csv(per_run_csv)
mean_df    = pd.read_csv(mean_csv)

n_runs = per_run_df["Run"].nunique()
print(f"Plotting {n_runs} individual fits  (P%={P_CHECK*100:.1f}, BS={BS_CHECK})")

fig, ax = plt.subplots(figsize=(11, 5))

for run_id, grp in per_run_df.groupby("Run"):
    grp_sorted = grp.sort_values("Batch_Number")
    ax.plot(grp_sorted["Batch_Number"], grp_sorted["Fitted_CE"],
            color="steelblue", alpha=0.12, linewidth=0.8)

ax.plot(mean_df["Batch_Number"], mean_df["Mean_Fit_CE"],
        color="red", linewidth=2.5, label=f"Saved mean  (n={n_runs} runs)")
ax.fill_between(mean_df["Batch_Number"],
                mean_df["Mean_Fit_CE"] - mean_df["Std_Fit_CE"],
                mean_df["Mean_Fit_CE"] + mean_df["Std_Fit_CE"],
                color="red", alpha=0.15, label="±1 std")

ax.axhline(np.log(10), color="gray", linestyle="--", linewidth=1)
ax.text(0, np.log(10) + 0.04, r"$\ln(10)$", fontsize=11, color="gray")

ax.set_xlabel("Batch Number")
ax.set_ylabel("CE Test")
ax.set_ylim(0, 2.7)
ax.set_title(f"Individual fitted curves + mean  —  P%={P_CHECK*100:.1f}  BS={BS_CHECK}")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell C — Numerical Cross-Check: Re-derive Mean and Compare

The strongest check. Re-computes the mean independently from the `per_run_fits` CSV
(groupby `Batch_Number`, take mean of `Fitted_CE`) and compares it to the saved `mean_fit` CSV.

**Expected result**: max absolute difference < 1e-10 (floating-point noise only).

If the difference is larger, it means the saved mean was computed on a different x-grid
than what is stored in the per-run CSV — pointing to the specific line in Cell 2 to investigate.

In [ ]:
# ── Cell C — Numerical cross-check ──────────────────────────────────────────
# Uses the same (P_CHECK, BS_CHECK) selected in Cell B.

# Re-derive mean from the per-run CSV
recomputed = (
    per_run_df
    .groupby("Batch_Number")["Fitted_CE"]
    .agg(recomp_mean="mean", recomp_std="std")
    .reset_index()
)

# Merge with saved mean on Batch_Number
check = mean_df.merge(recomputed, on="Batch_Number", how="inner")

diff_mean = (check["Mean_Fit_CE"] - check["recomp_mean"]).abs()
diff_std  = (check["Std_Fit_CE"]  - check["recomp_std"]).abs()

print(f"Batch points in saved mean_fit CSV : {len(mean_df)}")
print(f"Batch points in per_run_fits CSV   : {recomputed['Batch_Number'].nunique()}")
print(f"Matched batch points (inner join)  : {len(check)}")
print()
print(f"Max |saved_mean - recomputed_mean| : {diff_mean.max():.2e}")
print(f"Max |saved_std  - recomputed_std|  : {diff_std.max():.2e}")

THRESHOLD = 1e-6
if diff_mean.max() < THRESHOLD:
    print(f"\n  PASS — mean curves match within {THRESHOLD:.0e}")
else:
    print(f"\n  FAIL — mismatch exceeds {THRESHOLD:.0e}; investigate Cell 2 x-grid logic")

# ── Optional: plot the difference ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(check["Batch_Number"], diff_mean, color="tomato", linewidth=1.2)
axes[0].set_title("Absolute diff: saved mean vs recomputed mean")
axes[0].set_xlabel("Batch Number")
axes[0].set_ylabel("|difference|")
axes[0].grid(True, alpha=0.3)

axes[1].plot(check["Batch_Number"], diff_std, color="steelblue", linewidth=1.2)
axes[1].set_title("Absolute diff: saved std vs recomputed std")
axes[1].set_xlabel("Batch Number")
axes[1].set_ylabel("|difference|")
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"Cross-check  —  P%={P_CHECK*100:.1f}  BS={BS_CHECK}", y=1.02)
plt.tight_layout()
plt.show()

## Cell D — Full Cross-Check Across All (P%, BS)

Runs the Cell C check for every combination and prints a summary table.
Any row with `max_diff > 1e-6` flags a mismatch worth investigating.

In [ ]:
# ── Cell D — Full cross-check for all (P%, BS) ───────────────────────────────

results = []

for bs in BATCH_SIZES:
    for p in PRUNING_LEVELS:
        per_run_path = os.path.join(OUT_DIR, f"BS_{bs}", f"per_run_fits_p_{p}_bs_{bs}.csv")
        mean_path    = os.path.join(OUT_DIR, f"BS_{bs}", f"mean_fit_p_{p}_bs_{bs}.csv")

        if not os.path.isfile(per_run_path) or not os.path.isfile(mean_path):
            results.append({"P%": p*100, "BS": bs, "matched_pts": None,
                             "n_runs": None, "max_diff": None, "status": "MISSING CSV"})
            continue

        pr  = pd.read_csv(per_run_path)
        mf  = pd.read_csv(mean_path)

        recomp = (
            pr.groupby("Batch_Number")["Fitted_CE"]
            .mean()
            .reset_index()
            .rename(columns={"Fitted_CE": "recomp_mean"})
        )

        merged   = mf.merge(recomp, on="Batch_Number", how="inner")
        max_diff = (merged["Mean_Fit_CE"] - merged["recomp_mean"]).abs().max()
        n_runs   = pr["Run"].nunique()

        results.append({
            "P%":          p * 100,
            "BS":          bs,
            "n_runs":      n_runs,
            "matched_pts": len(merged),
            "max_diff":    max_diff,
            "status":      "PASS" if max_diff < 1e-6 else "FAIL"
        })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

failures = result_df[result_df["status"] == "FAIL"]
print(f"\nSummary: {len(failures)} failures out of {len(result_df)} combinations.")
if not failures.empty:
    print("Failures:")
    print(failures[["P%", "BS", "n_runs", "max_diff"]].to_string(index=False))